# Semana 9: Multiagentes con CrewAI en Kaggle

Adaptación de `Sem9_Course_MultiAgent_CrewAI.ipynb`. Conserva los dos ejemplos: planificación, escritura y edición de un blog sobre **Aborto**; atención y revisión de soporte para **DeepLearningAI**. Mantiene la misma API KEY de ejemplo y el modelo Groq `openai/gpt-oss-20b`.

Ejecutar todas las celdas en orden con **Internet activado**. No requiere reiniciar el kernel. La GPU T4 queda habilitada en metadata; las inferencias se ejecutan en Groq. Una clave de ejemplo solo produce respuestas si Groq la acepta.

Los resultados y el estado real de cada flujo se guardan en `/kaggle/working`. `run_summary.json` distingue éxito de errores aunque Kaggle termine el notebook. El blog original no incluye búsqueda web: sus afirmaciones no equivalen a noticias o hechos verificados. El soporte consulta documentación oficial; la memoria sigue desactivada, como en el original.


### 1. Instalar dependencias

Versiones fijadas; los requisitos van entre comillas para evitar que `>` se interprete como redirección de shell. Instalación antes de importar CrewAI.

In [ ]:
import os
os.environ['CREWAI_TELEMETRY_DISABLED'] = 'true'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['CREWAI_TRACING_ENABLED'] = 'false'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
%pip install -q "crewai[litellm,tools]==1.15.27" "litellm==1.89.2" "groq==0.37.1"

### 2. Compatibilidad con Groq

Conserva el ajuste de caché del notebook base, aplicado antes de cargar agentes. Evita enviar marcadores de caché no soportados por Groq.

In [ ]:
# Parche para funcionar en COLAB y trabajar con Groq

import crewai.llms.cache as _crewai_cache

_crewai_cache.mark_cache_breakpoint = lambda msg: msg

### 3. Configurar modelo

API KEY de ejemplo copiada sin cambios. Solo se imprime si está configurada; nunca su contenido.

In [ ]:
import os
from getpass import getpass
from crewai import Agent, Task, Crew, Process, LLM

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Ingresa GROQ_API_KEY: ")

# Seleccionamos un modelo de los que Groq dispone
MODEL = "openai/gpt-oss-20b"

print("GROQ_API_KEY configurada:", bool(os.environ.get("GROQ_API_KEY")))

llm = LLM(
    model=f"groq/{MODEL}",
    api_key=os.environ["GROQ_API_KEY"],
    temperature=0.2,
    max_tokens=1500,
    timeout=90,
    max_retries=2,
)

### 4. Preparar archivos y estado

Guarda cada tarea y el resultado final. Los errores quedan registrados sin revelar la API KEY; ambos ejemplos se intentan de forma independiente.

In [ ]:
import json
import platform
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path
from IPython.display import Markdown, display

OUTPUT_DIR = Path('/kaggle/working')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
summary = {
    'status': 'running',
    'started_at': datetime.now(timezone.utc).isoformat(),
    'model': f'groq/{MODEL}',
    'python': platform.python_version(),
    'packages': {name: version(name) for name in ('crewai', 'crewai-tools', 'litellm', 'groq')},
    'flows': {},
}

def save_summary():
    (OUTPUT_DIR / 'run_summary.json').write_text(
        json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8'
    )

async def run_flow(name, selected_crew, inputs):
    summary['flows'][name] = {'status': 'running', 'inputs': inputs}
    save_summary()
    try:
        result = await selected_crew.kickoff_async(inputs=inputs)
        assert result.raw.strip(), 'El flujo no produjo contenido.'
        final_file = OUTPUT_DIR / f'{name}.md'
        final_file.write_text(result.raw, encoding='utf-8')
        summary['flows'][name] = {
            'status': 'complete', 'inputs': inputs,
            'tasks_completed': len(result.tasks_output),
            'artifact': final_file.name,
        }
        print(f'{name}: complete; {len(result.tasks_output)} tareas; {final_file.name}')
        return result
    except Exception as exc:
        message = str(exc).replace(os.environ['GROQ_API_KEY'], '[API KEY omitida]')
        summary['flows'][name] = {
            'status': 'failed', 'inputs': inputs,
            'error_type': type(exc).__name__, 'error': message[:2000],
        }
        print(f'{name}: failed; {type(exc).__name__}: {message[:400]}')
        return None
    finally:
        save_summary()

save_summary()

### 5. Agentes y tareas del blog

Mantiene planificador, escritor y editor con sus instrucciones originales. Proceso secuencial y salidas por tarea.

In [ ]:
# Warning control
import warnings

warnings.filterwarnings("ignore")

from crewai import Agent, Task, Crew

planner = Agent(
    role="Planificador",
    goal="Planifica contenido adecuado para el {topico}",
    backstory="Tú estás trabajando en un blog sobre el tópico: {topico}. Debe recolectar información que ayude a la audiencia a comprender sobre algo y tomas decisiones informadas.",
    allow_delegation=False,  # evitar que se delege a otro agente la tarea
    verbose=False,
    max_iter=8,  # para ver como hace el razonamiento
    llm=llm,
)

writer = Agent(
    role="Escritor",
    goal="Escribir un artículo de opinión sobre el tópico: {topico}",
    backstory="Tú estás trabajando en escribir un artículo de opinión sobre el tópico: {topico}. Debes seguir las instrucciones del Planificador. El artículo debe tener opiniones objetivas.",
    allow_delegation=False,  # evitar que se delege a otro agente la tarea
    verbose=False,
    max_iter=8,  # para ver como hace el razonamiento
    llm=llm,
)

editor = Agent(
    role="Editor",
    goal="Editar un artículo de opinión y darle el estilo y formao d ela organización.",
    backstory="Tú eres un editor que recibe un artículo de opinión del escritor. Tú debes revisarlo y revisar que cumpla buenas prácticas de periodismo.",
    allow_delegation=False,  # evitar que se delege a otro agente la tarea
    verbose=False,
    max_iter=8,  # para ver como hace el razonamiento
    llm=llm,
)

plan = Task(
    description=(
        "1. Prioriza utilizar las últimas tendencias y noticias sobre {topico}.\n"
        "2. Identifica tu público objetivo considerando intereses.\n"
        "3. Desarrolla una pauta de contenido que incluya introducción, puntos claves y un llamado a la acción.\n"
    ),
    expected_output="Un documento de plan de contenidos",
    agent=planner,
)

write = Task(
    description=(
        "1. Use the content plan to craft a compelling "
        "blog post on {topico}.\n"
        "2. Incorporate SEO keywords naturally.\n"
        "3. Sections/Subtitles are properly named "
        "in an engaging manner.\n"
        "4. Ensure the post is structured with an "
        "engaging introduction, insightful body, "
        "and a summarizing conclusion.\n"
        "5. Proofread for grammatical errors and "
        "alignment with the brand's voice.\n"
    ),
    expected_output="A well-written blog post "
    "in markdown format, ready for publication, "
    "each section should have 2 or 3 paragraphs.",
    agent=writer,
)

edit = Task(
    description=(
        "Proofread the given blog post for "
        "grammatical errors and "
        "alignment with the brand's voice."
    ),
    expected_output="A well-written blog post in markdown format, "
    "ready for publication, "
    "each section should have 2 or 3 paragraphs.",
    agent=editor,
)


plan.output_file = str(OUTPUT_DIR / 'blog_plan.md')
write.output_file = str(OUTPUT_DIR / 'blog_draft.md')
edit.output_file = str(OUTPUT_DIR / 'blog_edited.md')
blog_crew = Crew(
    agents=[planner, writer, editor],
    tasks=[plan, write, edit],
    process=Process.sequential,
    verbose=False,
    memory=False,
    max_rpm=15,
)

### 6. Ejecutar blog

Ejecuta las tres tareas reales en Groq, conservando el tópico del notebook base.

In [ ]:
blog_result = await run_flow('blog', blog_crew, {'topico': 'Aborto'})

### 7. Mostrar blog

Presenta Markdown solo cuando Groq devolvió un resultado.

In [ ]:
if blog_result is not None:
    display(Markdown(blog_result.raw))
else:
    print('Blog sin resultado. Revisar run_summary.json.')

### 8. Agentes y herramientas de soporte

Mantiene representante y revisor. Scraping de documentación oficial de crews y memoria; elimina Serper y WebsiteSearch sin uso para evitar claves adicionales.

La URL se pasa como argumento al tool: un tool con URL fija genera un esquema vacío que Groq rechaza. Se valida el esquema antes de ejecutar.

Groq limita cada solicitud a 8.000 TPM en esta cuenta. El tool extrae el artículo sin navegación y devuelve hasta 4.500 caracteres por página, marcando el recorte. Las respuestas se limitan a 1.500 tokens.

In [ ]:
from crewai_tools import ScrapeWebsiteTool
from crewai_tools.security.safe_requests import safe_get
from bs4 import BeautifulSoup

def bounded_context(text, limit=4500):
    if len(text) <= limit:
        return text
    return text[:2500] + "\n[Contenido intermedio omitido por límite de tokens]\n" + text[-1900:]

assert len(bounded_context('x' * 20000)) <= 4500

class KaggleScrapeWebsiteTool(ScrapeWebsiteTool):
    """Conserva el tool de scraping con contexto acotado para Groq."""

    def _run(self, **kwargs):
        url = kwargs['website_url']
        allowed = {
            'https://docs.crewai.com/en/concepts/crews',
            'https://docs.crewai.com/en/concepts/memory',
        }
        if url not in allowed:
            return 'Consultar solo las dos URLs oficiales indicadas en la tarea.'
        page = safe_get(url, timeout=15, headers=self.headers)
        page.raise_for_status()
        soup = BeautifulSoup(page.text, 'html.parser')
        article = soup.find(id='content-area') or soup.find('main') or soup
        text = article.get_text(' ', strip=True)
        assert text.strip(), 'La documentación no devolvió contenido.'
        print(f'Documentación consultada: {url}; contexto: {min(len(text), 4500)} caracteres')
        return f'Fuente: {url}\n' + bounded_context(text)


support_agent = Agent(
    role="Senior Support Representative",
    goal="Be the most friendly and helpful " "support representative in your team",
    backstory=(
        "You work at crewAI (https://crewai.com) and "
        " are now working on providing "
        "support to {customer}, a super important customer "
        " for your company."
        "You need to make sure that you provide the best support!"
        "Make sure to provide full complete answers, "
        " and make no assumptions."
    ),
    allow_delegation=False,
    verbose=False,
    max_iter=8,
    llm=llm,
)

support_quality_assurance_agent = Agent(
    role="Support Quality Assurance Specialist",
    goal="Get recognition for providing the "
    "best support quality assurance in your team",
    backstory=(
        "You work at crewAI (https://crewai.com) and "
        "are now working with your team "
        "on a request from {customer} ensuring that "
        "the support representative is "
        "providing the best support possible.\n"
        "You need to make sure that the support representative "
        "is providing full"
        "complete answers, and make no assumptions."
    ),
    verbose=False,
    max_iter=8,
    allow_delegation=False,
    llm=llm,
)

docs_scrape_tool = KaggleScrapeWebsiteTool()
assert "website_url" in docs_scrape_tool.args_schema.model_json_schema()["properties"]

inquiry_resolution = Task(
    description=(
        "{customer} just reached out with a super important ask:\n"
        "{inquiry}\n\n"
        "{person} from {customer} is the one that reached out. "
        "Make sure to use everything you know "
        "to provide the best support possible."
        "You must strive to provide a complete "
        "and accurate response to the customer's inquiry."
    ),
    expected_output=(
        "A detailed, informative response to the "
        "customer's inquiry that addresses "
        "all aspects of their question.\n"
        "The response should include references "
        "to everything you used to find the answer, "
        "including external data or solutions. "
        "Ensure the answer is complete, "
        "leaving no questions unanswered, and maintain a helpful and friendly "
        "tone throughout."
    ),
    tools=[docs_scrape_tool],  # Defines las tools a utilizar a nivel de Task
    agent=support_agent,
)

quality_assurance_review = Task(
    description=(
        "Review the response drafted by the Senior Support Representative for {customer}'s inquiry. "
        "Ensure that the answer is comprehensive, accurate, and adheres to the "
        "high-quality standards expected for customer support.\n"
        "Verify that all parts of the customer's inquiry "
        "have been addressed "
        "thoroughly, with a helpful and friendly tone.\n"
        "Check for references and sources used to "
        " find the information, "
        "ensuring the response is well-supported and "
        "leaves no questions unanswered."
    ),
    expected_output=(
        "A final, detailed, and informative response "
        "ready to be sent to the customer.\n"
        "This response should fully address the "
        "customer's inquiry, incorporating all "
        "relevant feedback and improvements.\n"
        "Don't be too formal, we are a chill and cool company "
        "but maintain a professional and friendly tone throughout."
    ),
    agent=support_quality_assurance_agent,
)



inquiry_resolution.description += (
    "\nUse the website reading tool to consult both official sources: "
    "https://docs.crewai.com/en/concepts/crews and "
    "https://docs.crewai.com/en/concepts/memory. "
    "Pass website_url explicitly to the tool and cite the consulted URLs."
)
inquiry_resolution.output_file = str(OUTPUT_DIR / 'support_draft.md')
quality_assurance_review.output_file = str(OUTPUT_DIR / 'support_reviewed.md')
support_crew = Crew(
    agents=[support_agent, support_quality_assurance_agent],
    tasks=[inquiry_resolution, quality_assurance_review],
    process=Process.sequential,
    verbose=False,
    memory=False,
    max_rpm=15,
)

### 9. Ejecutar soporte

Consulta original de Robert, DeepLearningAI, sobre creación, kickoff y memoria.

In [ ]:
inputs = {
    "customer": "DeepLearningAI",
    "person": "Robert",
    "inquiry": "I need help with setting up a Crew and kicking it off. How can I add memory to my crew? Can you provide guidance?",
}


support_result = await run_flow('support', support_crew, inputs)

### 10. Mostrar soporte

Respuesta revisada; no envía mensajes al cliente.

In [ ]:
if support_result is not None:
    display(Markdown(support_result.raw))
else:
    print('Soporte sin resultado. Revisar run_summary.json.')

### 11. Verificar resultados

Éxito requiere los dos flujos completos y cinco tareas. Lista archivos recuperables desde Kaggle.

In [ ]:
states = [summary['flows'][name]['status'] for name in ('blog', 'support')]
summary['status'] = 'complete' if all(state == 'complete' for state in states) else 'failed'
summary['finished_at'] = datetime.now(timezone.utc).isoformat()
if summary['status'] == 'complete':
    assert summary['flows']['blog']['tasks_completed'] == 3
    assert summary['flows']['support']['tasks_completed'] == 2
    for name in ('blog', 'support'):
        assert (OUTPUT_DIR / summary['flows'][name]['artifact']).stat().st_size > 0
save_summary()
print(json.dumps(summary, ensure_ascii=False, indent=2))
print('Archivos:', ', '.join(path.name for path in sorted(OUTPUT_DIR.glob('*.md'))))